In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data

In [ ]:
from data import download_tickers_history
from data.constants import TRADING_DAYS_PER_YEAR

# set the date range for the historic data
start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers).dropna()

start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['^GSPC']
sp500_history = download_tickers_history(start_date, end_date, tickers).dropna()

history.head()


## Backtesting logic
Backtesting of a portfolio strategy is a historical simulation of the workings of a placeholder in time to test how the weight distribution would have performed on real out-of-sample data.

The portfolio cannot be optimized once across the entire story - this would create a critical look-ahead bias.
Instead, the simulation moves over time step by step:

1. **Time $t$ (Rebalancing Date)**: we take a line from the history strictly to $t$ (for example, last 504 trading days).
2. **Parameter evaluation**: we calculate the cost of $\Sigma_{t+1}$ via GARCH/EGARCH or Ledoit-Wolf. We estimate the expected returns of $\mu_{t+1}$ via Black-Litterman.
3. **Optimization**: find the optimal vector for $w_t *$ target weights (Max Sharpe, Min Volatility, etc.).
4. **Out-of-Sample execution**: "freeze" weights $w_t *$ and apply them to real price movements from the moment of $t$ until the next rebalancing date of $t +  Delta t$ (for example, on the 21st trading day).
5. **Window shift**: go to $t + \Delta t$ and repeat the procedure.

#### What to take into account for correct basket
To reflect real trade, the basket has four key components:
* **Frequency of rebalancing ($\Delta t$)**
    * Daily: too much trading; we will lose all in profit by commissions.
    * Monthly (~21 trading day): Standard for medium-term portfolio strategies.
* **Transaction costs & slippage**
    
    When changing weights from $w_{t-1}$ to $w_t$, the portfolio makes turnover (Turnover):
    $$\text{Turnover}_t = \sum_{i=1}^N \vert{}w_{i, t} - w_{i, t^-}\vert{},$$
    Each deal is covered by a broker’s commission and through capture (usually $0.05% - 0.1% of the deal’s value).
* **Weight drift**
    
    Between rebalancing days, asset prices change unevenly. The share of rising stocks naturally increases and falling shares decline within a holding period.
* **Benchmark for comparison**
    * Strategy results are necessarily overlaid on charts: **S&P 500 (^GSPC)** - passive market.
    * **$1/N$ (Equal-Weighted)** - naive diversification with similar frequency of rebalancing.

In [ ]:
from src.backtest import perform_backtesting

lookback_window = 504
rebalancing_period = 21 # 21-trading day

portfolio_value = 10000.0 # starting capital = 10,000 $
broker_commission = 0.0005 # 0.05%

backtest_res_df = perform_backtesting(
    history,
    portfolio_value,
    'GARCH_SHARPE',
    'BLACK_LITTERMAN',
    sp500_history,
    lookback_window,
    rebalancing_period,
    broker_commission,
    'FIXED'
)

backtest_res_df.head()


## Visualize backtesting data

#### Cumulative Equity Curve
Shows whether a complex algorithmic platform overtakes the simple passive market and naive (equal-weighted) diversification at a distance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Trader's portfolio
date = pd.to_datetime(backtest_res_df.index)
equity = backtest_res_df["portfolio_value"]

ax.plot(
    date,
    equity,
    color="darkgreen",
    linewidth=2,
    label="Portfolio Cumulative Equity",
)

# Equals weights
eq_equity = backtest_res_df["equal_portfolio_value"]

ax.plot(
    date,
    eq_equity,
    color="darkorange",
    linewidth=1,
    label="Equal Weights Portfolio Cumulative Equity",
)

# S&P 500
sp500_equity = backtest_res_df["sp500_value"]

ax.plot(
    date,
    sp500_equity,
    color="darkblue",
    linewidth=1,
    label="S&P 500 Cumulative Equity",
)

# Title and labels
plt.title('Cumulative Equity Curve', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Returns ($)', fontsize=12)

# Legend box showing what each color curve represents
ax.legend(
    title="Curve Legend",
    title_fontsize=11,
    loc="upper left",
    frameon=True,
    facecolor="#ffffff",
    edgecolor="black",
    framealpha=0.95,
    fontsize=10,
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


#### Underwater Drawdown Chart
Shows strategy stress test. The optimized portfolio should have a significantly lower depth of flow and faster recovery (Recovery Time) than the _S&P 500_ during crises.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Trader's portfolio
date = pd.to_datetime(backtest_res_df.index)
equity = backtest_res_df["portfolio_value"]
drawdown = (equity - equity.cummax()) / equity.cummax()

ax.plot(
    date,
    drawdown,
    color="darkgreen",
    linewidth=2,
    label="Portfolio Underwater Drawdown",
)

ax.axhline(0, color="gray", linestyle="-", linewidth=1, alpha=1)

# Equal weights portfolio
eq_equity = backtest_res_df["equal_portfolio_value"]
eq_drawdown = (eq_equity - eq_equity.cummax()) / eq_equity.cummax()

ax.plot(
    date,
    eq_drawdown,
    color="darkorange",
    linewidth=1,
    label="Equal Weights Portfolio Underwater Drawdown",
)

# S&P 500
sp500_equity = backtest_res_df["sp500_value"]
sp500_drawdown = (sp500_equity - sp500_equity.cummax()) / sp500_equity.cummax()

ax.plot(
    date,
    sp500_drawdown,
    color="darkblue",
    linewidth=1,
    label="S&P 500 Underwater Drawdown",
)

# Title and labels
plt.title('Underwater Drawdown Curve', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Returns ($)', fontsize=12)

# Legend box showing what each color curve represents
ax.legend(
    title="Curve Legend",
    title_fontsize=11,
    loc="lower left",
    frameon=True,
    facecolor="#ffffff",
    edgecolor="black",
    framealpha=0.95,
    fontsize=10,
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


#### Weights History
Shows model stability and adequacy:
* If weights change smoothly from month to month, the model is stable.
* If the weights are chaotically raised from $0% to $100% each step - too much noise in the optimizer, transaction fees will eat up all the profits.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Trader's portfolio
close = history.xs("Close", axis=1, level="Price")
weights_df = pd.DataFrame(
    (backtest_res_df["weights"] * 100).tolist(),
    index=backtest_res_df.index,
    columns=close.columns
)
date = weights_df.index
ax.stackplot(
    date,
    weights_df.T,
    linewidth=2,
    labels=weights_df.columns,
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1],
    labels[::-1],
    title="Assets",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True
)

plt.title('Weights History Stacked Area', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Weights (%)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


#### Rolling Sharpe Ratio, 6M / 1Y
Shows the stability of alpha generation over time. Good strategy keeps Rolling Sharpe consistently above zero, without falling into deep downsides during adjustments.

In [ ]:
from src.portfolio import get_risk_free_rate

fig, ax = plt.subplots(figsize=(12, 6))

window = 126 # 126 days = 6 trading months
risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

# Trader's portfolio
date = pd.to_datetime(backtest_res_df.index)
portfolio_ret = backtest_res_df["daily_return"].rolling(window)
portfolio_rolling_sharpe = ((portfolio_ret.mean() * TRADING_DAYS_PER_YEAR - risk_free_rate) / (portfolio_ret.std() * np.sqrt(TRADING_DAYS_PER_YEAR)))

ax.plot(
    date,
    portfolio_rolling_sharpe,
    color="darkgreen",
    linewidth=2,
    label="Portfolio Rolling Sharpe",
)

ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(1.0, color="darkblue", linestyle=":", linewidth=1, alpha=0.5, label="Sharpe = 1.0 (Good)")

# Equal weights portfolio
eq_portfolio_ret = backtest_res_df["equal_portfolio_daily_return"].rolling(window)
eq_rolling_sharpe = ((eq_portfolio_ret.mean() * TRADING_DAYS_PER_YEAR - risk_free_rate) / (eq_portfolio_ret.std() * np.sqrt(TRADING_DAYS_PER_YEAR)))

ax.plot(
    date,
    eq_rolling_sharpe,
    color="darkorange",
    linewidth=1,
    label="Equal Weights Portfolio Rolling Sharpe",
)

# S&P 500
out_of_sample_sp500_df = sp500_history[sp500_history.index.isin(date)]
sp500_close = out_of_sample_sp500_df.xs("Close", axis=1, level="Price")
sp500_ret = (sp500_close / sp500_close.shift(1) - 1).rolling(window)
sp500_rolling_sharpe = ((sp500_ret.mean() * TRADING_DAYS_PER_YEAR - risk_free_rate) / (sp500_ret.std() * np.sqrt(TRADING_DAYS_PER_YEAR)))

ax.plot(
    date,
    sp500_rolling_sharpe,
    color="darkblue",
    linewidth=1,
    label="S&P 500 Underwater Rolling Sharpe",
)

# Title and labels
plt.title('Rolling Sharpe Ratio', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Returns ($)', fontsize=12)

# Legend box showing what each color curve represents
ax.legend(
    title="Curve Legend",
    title_fontsize=11,
    loc="lower left",
    frameon=True,
    facecolor="#ffffff",
    edgecolor="black",
    framealpha=0.95,
    fontsize=10,
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
